# 1. Setup & Installation
Install dependencies including `bitsandbytes`, `accelerate`, and translation models.


In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 beautifulsoup4 transformers torch sentence-transformers faiss-cpu accelerate spacy networkx pyvis bitsandbytes -q
!python -m spacy download en_core_web_sm -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 114.6 MB/s eta 0:00:00
✔ Download and installation successful
You can n

# 2. Google Drive Mounting
Store the database, raw policy documents, FAISS index, and graphs permanently in Google Drive.


In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Setup Persistent Directories
APP_DIR = "/content/drive/MyDrive/PolicyNav"
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'documents'), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'graphs'), exist_ok=True)
os.environ['APP_DIR'] = APP_DIR

print(f"✅ Persistent App Directory set to: {APP_DIR}")
print(f"👉 Please upload your policy PDFs/files directly to {os.path.join(APP_DIR, 'documents')} in your Google Drive.")



Mounted at /content/drive
✅ Persistent App Directory set to: /content/drive/MyDrive/PolicyNav
👉 Please upload your policy PDFs/files directly to /content/drive/MyDrive/PolicyNav/documents in your Google Drive.


# 3. Write Core Modules to File System


In [ ]:
%%writefile db.py
import sqlite3, bcrypt, datetime, os

DB_NAME = os.path.join(os.environ.get('APP_DIR', '.'), "users.db")

def _get_conn():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    conn = _get_conn(); c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS users (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS activity_history (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, activity_type TEXT, input_text TEXT, output_text TEXT, timestamp TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, section TEXT, rating INTEGER, comments TEXT, timestamp TEXT)")
    conn.commit(); conn.close()

def log_activity(email, activity_type, input_text, output_text):
    conn = _get_conn(); c = conn.cursor()
    c.execute("INSERT INTO activity_history (email, activity_type, input_text, output_text, timestamp) VALUES (?, ?, ?, ?, ?)",
              (email, activity_type, input_text, output_text, datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit(); conn.close()

def get_user_activity(email):
    conn = _get_conn(); c = conn.cursor()
    c.execute("SELECT activity_type, input_text, output_text, timestamp FROM activity_history WHERE email = ? ORDER BY timestamp DESC LIMIT 50", (email,))
    data = c.fetchall(); conn.close(); return data

def submit_feedback(email, section, rating, comments=""):
    conn = _get_conn(); c = conn.cursor()
    c.execute("INSERT INTO feedback (email, section, rating, comments, timestamp) VALUES (?, ?, ?, ?, ?)",
              (email, section, rating, comments, datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit(); conn.close()

def register_user(email, password):
    conn = _get_conn(); c = conn.cursor()
    try:
        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)",
                  (email, bcrypt.hashpw(password.encode(), bcrypt.gensalt()), datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
        conn.commit(); return True
    except: return False
    finally: conn.close()

def authenticate_user(email, password):
    conn = _get_conn(); c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone(); conn.close()
    return True if data and bcrypt.checkpw(password.encode(), data[0]) else False



Writing db.py


In [ ]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text)
        }

    def get_all_metrics(self):
        return self.metrics

    def get_audience(self):
        score = self.metrics["Flesch Reading Ease"]
        if score >= 90: return "5th Grader"
        elif score >= 60: return "8th & 9th Grader"
        elif score >= 30: return "College Student"
        else: return "College Graduate / Technical"



Writing readability.py


In [ ]:
%%writefile vector_store.py
import os, glob, pickle, faiss, PyPDF2
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

APP_DIR = os.environ.get('APP_DIR', '.')
INDEX_PATH = os.path.join(APP_DIR, "faiss_index.bin")
META_PATH = os.path.join(APP_DIR, "faiss_meta.pkl")

# We only load it once cached in Streamlit
_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _embedder

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        for page in PyPDF2.PdfReader(pdf_path).pages: text += page.extract_text() + "\n"
    except: pass
    return text

def ingest_documents(docs_dir):
    if not os.path.exists(docs_dir): return 0

    # Load existing metdata to avoid re-ingesting
    existing_metadata = []
    if os.path.exists(META_PATH):
        try:
            with open(META_PATH, 'rb') as f: existing_metadata = pickle.load(f)
        except: pass

    existing_filenames = set([d['filename'] for d in existing_metadata])

    files = glob.glob(os.path.join(docs_dir, "*"))
    new_chunks = []
    new_metadata = []

    for filepath in files:
        filename = os.path.basename(filepath)
        if filename in existing_filenames: continue # Skip already ingested!

        print(f"Parsing new document: {filename}")
        text = ""
        if filepath.lower().endswith(".pdf"): text = extract_text_from_pdf(filepath)
        elif filepath.lower().endswith((".htm", ".html")): text = BeautifulSoup(open(filepath, 'r').read(), 'html.parser').get_text(separator=' ')
        elif filepath.lower().endswith(".txt"): text = open(filepath, 'r').read()

        if text.strip():
            for i in range(0, len(text), 1500):
                chunk = text[i:i+1500]
                if len(chunk) > 50:
                    new_chunks.append(chunk)
                    new_metadata.append({"filename": filename, "content": chunk})

    if not new_chunks: return 0 # Nothing new to ingest

    embedder = get_embedder()
    embeddings = embedder.encode(new_chunks, convert_to_numpy=True)

    if os.path.exists(INDEX_PATH):
        index = faiss.read_index(INDEX_PATH)
    else:
        index = faiss.IndexFlatL2(embeddings.shape[1])

    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

    final_metadata = existing_metadata + new_metadata
    with open(META_PATH, 'wb') as f: pickle.dump(final_metadata, f)
    return len(new_chunks)

def search_documents(query, top_k=5):
    if not os.path.exists(INDEX_PATH) or not os.path.exists(META_PATH): return []
    try:
        embedder = get_embedder()
        index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, 'rb') as f: metadata = pickle.load(f)
        distances, indices = index.search(embedder.encode([query], convert_to_numpy=True), top_k)

        # Deduplicate to try to get broader file sources rather than just 5 chunks from the exact same page
        results = []
        seen_files = set()
        for idx in indices[0]:
            if idx != -1 and idx < len(metadata):
                doc = metadata[idx]
                fname = doc['filename']
                if list(seen_files).count(fname) < 2: # Max 2 chunks from same file allowed
                    results.append(doc)
                    seen_files.add(fname)
        return results
    except: return []

def get_all_documents():
    if not os.path.exists(META_PATH): return []
    with open(META_PATH, 'rb') as f: return pickle.load(f)



Writing vector_store.py


In [ ]:
%%writefile knowledge_graph.py
import spacy, networkx as nx, os
from pyvis.network import Network

try: nlp = spacy.load("en_core_web_sm")
except: nlp = None

APP_DIR = os.environ.get('APP_DIR', '.')

def build_graph_from_documents(docs):
    G = nx.Graph()
    max_docs = min(len(docs), 50)
    if not nlp: return G

    for i in range(max_docs):
        text = docs[i]['content'][:1000]
        doc = nlp(text)
        entities = [ent.text.strip() for ent in doc.ents if ent.label_ in ['ORG', 'GPE', 'LAW', 'PERSON'] and len(ent.text.strip()) > 2]
        source_node = f"Doc: {docs[i]['filename']}"
        G.add_node(source_node, title="Document", color="blue")
        for ent in entities:
            G.add_node(ent, title="Entity", color="green")
            G.add_edge(source_node, ent)
    return G

def generate_interactive_graph(docs):
    if not docs: return None
    G = build_graph_from_documents(docs)
    net = Network(height="600px", width="100%", bgcolor="#0e1117", font_color="white", notebook=False)
    net.from_nx(G)
    output_path = os.path.join(APP_DIR, "graphs", "policy_kg.html")
    net.save_graph(output_path)
    return output_path



Writing knowledge_graph.py


In [ ]:
%%writefile nlp_engine.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import vector_store

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRANSLATOR_ID = "facebook/nllb-200-distilled-600M"

# Globally cached in Streamlit
model = None
tokenizer = None
translator_model = None
translator_tokenizer = None

def init_model():
    global model, tokenizer, translator_model, translator_tokenizer
    if model is None:
        print(f"Loading {MODEL_ID} in 4-bit Quantization via BitsAndBytes...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=quantization_config)

        print(f"Loading {TRANSLATOR_ID} for extreme speed translation...")
        translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ID)
        translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_ID, device_map="auto")
        print("Models Loaded successfully!")

# BCP-47 Code mappings for NLLB-200
LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

def translate_fast(text, source_lang, target_lang):
    if source_lang == target_lang: return text

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    translator_tokenizer.src_lang = src_code
    inputs = translator_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(translator_model.device)

    # Fix for AttributeError: Use convert_tokens_to_ids instead of lang_code_to_id
    tgt_token_id = translator_tokenizer.convert_tokens_to_ids(tgt_code)
    outputs = translator_model.generate(**inputs, forced_bos_token_id=tgt_token_id, max_length=512)
    return translator_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_english_response(prompt_text):
    if model is None: init_model()
    messages = [
        {"role": "system", "content": "You are the Public Policy Compass AI utilizing RAG. Give accurate, highly concise answers in English based closely on the context."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, max_new_tokens=250, temperature=0.2)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def answer_policy_question(native_question, target_lang="English", simplify=False):
    # 1. Bi-directional matching: Translate their Native question into English FIRST so FAISS can search the Vector DB properly
    english_query = translate_fast(native_question, target_lang, "English")

    # 2. Search FAISS Vectors (Multiple distinct sources)
    docs = vector_store.search_documents(english_query, top_k=5)
    if not docs: docs_context = "No relevant policies found in Vector DB."
    else: docs_context = "\n\n".join([f"Snippet from {d['filename']}:\n{d['content']}" for d in docs])

    prompt = f"Context:\n{docs_context}\n\nQuestion: {english_query}\nAnswer the question using the context. Be direct."
    if simplify: prompt += " Simplify the policy language so a middle-school student can immediately understand it."

    # 3. LLM Generates English Answer
    eng_ans = generate_english_response(prompt)

    # 4. NLLB perfectly translates English Answer back to Native Language
    final_ans = translate_fast(eng_ans, "English", target_lang)
    return final_ans, docs

def summarize_document(text, target_lang="English"):
    eng_sum = generate_english_response(f"Summarize this policy into 3 highly concise bullet points:\n\n{text[:3000]}")
    return translate_fast(eng_sum, "English", target_lang)



Writing nlp_engine.py


In [ ]:
%%writefile app.py
import streamlit as st
import os, time, db, vector_store, nlp_engine, knowledge_graph
from streamlit_option_menu import option_menu
import plotly.express as px
import pandas as pd

# 🚀 Ultra-fast Cold Start: Only Load Models!
# (Vectorization is already handled by Colab before Streamlit boots)
@st.cache_resource
def load_and_cache_models():
    db.init_db()
    nlp_engine.init_model()
    vector_store.get_embedder()
    return True

APP_DIR = os.environ.get('APP_DIR', '.')

st.set_page_config(page_title="Public Policy Compass UI", page_icon="🏛️", layout="wide")
st.markdown("""
<style>
.stApp { background-color: #0b0f19; color: #e2e8f0; }
.css-1d391kg { background-color: #1a202c; } /* Sidebar */
div[data-testid="stChatMessage"] { background-color: #1e293b; border-radius: 12px; border: 1px solid #334155; padding: 15px; margin-bottom: 10px; }
.stButton>button { border-radius: 8px; font-weight: bold; }
</style>
""", unsafe_allow_html=True)

with st.spinner("Initializing AI Core & Checking Google Drive..."):
    load_and_cache_models()

if 'user' not in st.session_state: st.session_state['user'] = None
if 'page' not in st.session_state: st.session_state['page'] = 'login'

def switch_page(p): st.session_state['page'] = p; st.rerun()

# --- REUSABLE UI COMPONENTS ---
def render_feedback_ui(section_name, generated_text, unique_key):
    with st.expander(f"📝 Provide Feedback for {section_name}"):
        col1, col2 = st.columns([1, 4])
        with col1:
            rating = st.radio("Rating (1-5)", [1, 2, 3, 4, 5], horizontal=True, key=f"r_{unique_key}")
        with col2:
            comments = st.text_input("Comments (optional)", key=f"c_{unique_key}")

        if st.button("Submit Feedback", key=f"fb_{unique_key}"):
            db.submit_feedback(st.session_state['user'], section_name, rating, comments)
            st.toast("✅ Thank you for your feedback!")
# ------------------------------

def login_page():
    st.markdown("<h1 style='text-align: center; color: #38bdf8;'>🏛️ Policy Compass V8 Pro</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; color: #94a3b8;'>Extreme Speed Multilingual RAG via NLLB & Qwen-1.5B 4-bit</p>", unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1,2,1])
    with col2:
        with st.container(border=True):
            email = st.text_input("Email", placeholder="citizen@india.in")
            pwd = st.text_input("Password", type="password")
            if st.button("Access Dashboard", type="primary", use_container_width=True):
                if db.authenticate_user(email, pwd): st.session_state['user'] = email; st.rerun()
                else: st.error("❌ Authentication Failed")
        st.button("Register New Account", on_click=lambda: switch_page("register"), use_container_width=True)

def register_page():
    st.markdown("<h2 style='text-align: center; color: #38bdf8;'>Create Account</h2>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1,2,1])
    with col2:
        with st.container(border=True):
            email = st.text_input("Work/Personal Email")
            pwd = st.text_input("Secure Password", type="password")
            if st.button("Complete Registration", type="primary", use_container_width=True):
                if db.register_user(email, pwd): st.success("Created! Logging in..."); time.sleep(1); switch_page("login")
                else: st.error("Email already in database.")
        st.button("Back to Login", on_click=lambda: switch_page("login"), use_container_width=True)

def rag_search_tab():
    st.markdown("<h2 style='color: #38bdf8;'>🔍 Advanced RAG Search</h2>", unsafe_allow_html=True)
    st.write("Query the Google Drive Vector Database directly. AI synthesizes answers only from policy context.")

    c1, c2 = st.columns([3, 1])
    with c1: target_lang = st.selectbox("Response Dialect (NLLB Pipeline):", ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"])
    with c2: simplify = st.toggle("🧠 Simplify Jargon")
    st.divider()

    if "rag_chat" not in st.session_state: st.session_state.rag_chat = []

    for i, msg in enumerate(st.session_state.rag_chat):
        with st.chat_message(msg["role"]): st.markdown(msg["content"])

    if prompt := st.chat_input("Ask a policy question..."):
        st.session_state.rag_chat.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)

        with st.chat_message("assistant"):
            with st.spinner(f"Bi-Directional Vector Lookup & NLLB Extracting > Translating..."):
                t1 = time.time()
                ans, docs = nlp_engine.answer_policy_question(prompt, target_lang=target_lang, simplify=simplify)
                t2 = time.time()

                final_txt = f"**{ans}**\n\n---\n*Inference: {round(t2-t1,2)}s | 📚 Sources: {', '.join(list(set([d['filename'] for d in docs]))) if docs else 'None'}*"
                st.markdown(final_txt)

                db.log_activity(st.session_state['user'], "RAG Search", prompt, final_txt)
                st.session_state.rag_chat.append({"role": "assistant", "content": final_txt})
                st.rerun()

    st.divider()
    render_feedback_ui("RAG Search", "General Page Feedback", "rag_global")

def summarization_tab():
    st.markdown("<h2 style='color: #38bdf8;'>📝 Document Summarizer</h2>", unsafe_allow_html=True)
    st.write("Upload a PDF explicitly, or paste raw text to translate and summarize.")

    col1, col2 = st.columns([1, 1])

    with col1:
        uploaded_file = st.file_uploader("Upload Policy PDF or TXT", type=["pdf", "txt"])
        txt = ""
        if uploaded_file is not None:
            if uploaded_file.name.endswith('.pdf'):
                import PyPDF2
                try:
                    for page in PyPDF2.PdfReader(uploaded_file).pages: txt += page.extract_text() + "\n"
                except: pass
            else: txt = uploaded_file.read().decode("utf-8")
            st.success("File Processed successfully!")

        txt_area = st.text_area("Or Paste Raw Policy Text:", value=txt, height=200)

    with col2:
        lang = st.selectbox("Summary Output Translation:", ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"])
        if st.button("Generate Summary", type="primary") and txt_area:
            with st.spinner("Qwen -> NLLB processing pipeline..."):
                summary = nlp_engine.summarize_document(txt_area, target_lang=lang)
                st.info(summary)
                db.log_activity(st.session_state['user'], "Summarization", "Document Uploaded", summary)

    st.divider()
    render_feedback_ui("Summarization", "General Page Feedback", "sum_global")

def readability_tab():
    st.markdown("<h2 style='color: #38bdf8;'>📈 Readability Metrics</h2>", unsafe_allow_html=True)

    txt = st.text_area("Analyze Text Complexity:", height=150)
    if st.button("Calculate Core Metrics", type="primary") and txt:
        import readability
        analyzer = readability.ReadabilityAnalyzer(txt)
        metrics = analyzer.get_all_metrics()

        st.success(f"**Target Audience Computed:** {analyzer.get_audience()}")
        db.log_activity(st.session_state['user'], "Readability", "Text Evaluated", f"Target: {analyzer.get_audience()}")

        df = pd.DataFrame(dict(r=list(metrics.values()), theta=list(metrics.keys())))
        fig = px.line_polar(df, r='r', theta='theta', line_close=True, template="plotly_dark")
        fig.update_traces(fill='toself', line_color='#38bdf8')
        st.plotly_chart(fig, use_container_width=True)

    st.divider()
    render_feedback_ui("Readability", "General Page Feedback", "read_global")

def graph_tab():
    st.markdown("<h2 style='color: #38bdf8;'>🕸️ Named Entity Graph</h2>", unsafe_allow_html=True)
    st.write("Extracting Orgs, Laws, and Persons directly from Google Drive Docs via spaCy NER.")

    docs = vector_store.get_all_documents()
    if not docs: st.warning("No documents exist in Google Drive."); return

    if st.button("🔄 Render Interactive Topology", type="primary"):
        with st.spinner("Computing Graph Layout..."):
            path = knowledge_graph.generate_interactive_graph(docs)
            if path:
                with open(path, 'r', encoding='utf-8') as f:
                    import streamlit.components.v1 as components
                    components.html(f.read(), height=650)
                db.log_activity(st.session_state['user'], "Knowledge Graph", "Generated Graph", "Success")

    st.divider()
    render_feedback_ui("Knowledge Graph", "General Page Feedback", "kg_global")

def overall_history_tab():
    st.markdown("<h2 style='color: #38bdf8;'>📜 Global Activity History</h2>", unsafe_allow_html=True)
    st.write("All your RAG searches, summarizations, and metrics from Google Drive.")

    activities = db.get_user_activity(st.session_state['user'])
    if not activities:
        st.info("No activity found for your account yet.")
        return

    df = pd.DataFrame(activities, columns=["App Section", "Your Input", "AI Output", "Timestamp"])
    st.dataframe(df, use_container_width=True, hide_index=True)

if st.session_state['user']:
    with st.sidebar:
        st.markdown(f"**User:** `{st.session_state['user']}`")
        opts = ["🔍 RAG Search", "📝 Summarization", "📈 Readability", "🕸️ Knowledge Graph", "📜 Global History"]
        selected = option_menu(None, opts, default_index=0, styles={"nav-link-selected": {"background-color": "#38bdf8", "color": "#000000"}})
        st.divider()
        st.caption("💾 SQLite synced to Drive.")
        if st.button("Log Out"): st.session_state['user'] = None; st.rerun()

    if selected == "🔍 RAG Search": rag_search_tab()
    elif selected == "📝 Summarization": summarization_tab()
    elif selected == "📈 Readability": readability_tab()
    elif selected == "🕸️ Knowledge Graph": graph_tab()
    elif selected == "📜 Global History": overall_history_tab()
else:
    if st.session_state['page'] == 'login': login_page()
    elif st.session_state['page'] == 'register': register_page()



Writing app.py


# 4. Auto-Ingest PDFs to Vector Database
Scan Google Drive for PDFs and convert them into FAISS embeddings *before* starting the UI so Streamlit boots instantly.


In [ ]:
import os, vector_store
# Ensure the model is loaded first so we don't duplicate memory
try: _ = vector_store.get_embedder()
except: pass

docs_dir = os.path.join(APP_DIR, 'documents')
print("🔍 Scanning Google Drive for new PDF policies...")
new_chunks = vector_store.ingest_documents(docs_dir)

if new_chunks > 0:
    print(f"✅ Auto-ingested {new_chunks} new text chunks into the Vector Database!")
else:
    print("⚡ Vector Database is already up to date. Skipping ingestion. (Fast Boot)")



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔍 Scanning Google Drive for new PDF policies...
Parsing new document: PMJDY_dhan-accounts-claims-govt.pdf
Parsing new document: PMJDY_hindi.pdf
Parsing new document: PMJDY_English.pdf
Parsing new document: PMJDY_2Schemes-most-visible-PM-Yojanas.pdf
Parsing new document: PMAY_Urban_Innovative%20Technologies%20.pdf
Parsing new document: PMAY_Urban_PMAY_1_crore_Flyer.pdf
Parsing new document: PMAY_Urban_PMAY Angikaar Flyer_29Aug_B.pdf
Parsing new document: PMAY_Urban_ARH-EOI.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-December-2025.pdf
Parsing new document: JalJeevanMission_Jal-Jeevan-Samvad-November-2025.pdf
⚡ Vector Database is already up to date. Skipping ingestion. (Fast Boot)


# 5. Expose UI via Ngrok
Execute Streamlit.


In [ ]:
import os, subprocess, time
from google.colab import userdata
from pyngrok import ngrok

# Explicitly ensure App Dir mapping is preserved to the shell
os.environ['APP_DIR'] = APP_DIR
ngrok_token = userdata.get('NGROK_AUTHTOKEN')

if ngrok_token:
    ngrok.set_auth_token(ngrok_token); ngrok.kill()
    process = subprocess.Popen(['streamlit', 'run', 'app.py'], env=os.environ.copy()); time.sleep(6)

    print(f"\n🔗 RAG Streamlit URL: {ngrok.connect(8501).public_url}\n")
    print("🛑 To shutdown Streamlit gracefully, press the Stop button on this Cell, or press ENTER below.")
    try: input("Press ENTER to stop...\n")
    except: pass
    finally: process.terminate(); ngrok.kill(); print("Stopped.")
else: print("❌ Ngrok Token missing from Colab secrets.")




🔗 RAG Streamlit URL: https://electrometrically-syntonous-jeanetta.ngrok-free.dev

🛑 To shutdown Streamlit gracefully, press the Stop button on this Cell, or press ENTER below.
Press ENTER to stop...

Stopped.
